# Shared data parsing and preparation

Once you have the appropriate licences, obtain and format the input files
following the [data guide](README.md). This notebook prepares the shared tables
for the remaining experiments using functions in
[`src/dtpkg/preparation.py`](../src/dtpkg/preparation.py).
See the [setup instructions](../README.md) for dependencies.

DrugBank supplies drug information, documented DDIs, drug–target associations
(DPI), and drug-level MeSH assignments. UniProt supplies protein–protein
interactions (PPI), NLM supplies the MeSH descriptor hierarchy, and Zheng et al.'s
supplement supplies reliable-negative reference pairs. The
[data guide](README.md#data-sources) lists releases and downloads.

Supply the DrugBank and MeSH XML files and the negative-pair workbook. Format
the DrugBank DPI records as a CSV with `db_id,targets`, and the UniProt PPI records
as a CSV with `uniprot_id,interactions`, following the
[examples and column definitions](README.md#input-formats).
Input filenames are your choice; set their paths below. DrugBank records and
derived datasets remain local.


## Input and output paths

Replace `...` with your own quoted paths. The input files can remain at their
current locations; `DATA_DIR` is the destination for the parsed datasets.
Use a new `DATA_DIR` when changing the input files, then enable `RUN_PREPARATION`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Output folder for the shared private datasets
DRUGBANK_XML = ...  # Downloaded DrugBank all-drug XML
MESH_XML = ...  # Downloaded desc2025.xml
NEGATIVE_SOURCE = ...  # Zheng Additional file 2 XLSX, or a candidate-pair CSV
DPI_CSV = ...  # DrugBank DPI records formatted as db_id,targets CSV
PPI_CSV = ...  # UniProt PPI records formatted as uniprot_id,interactions CSV
RUN_PREPARATION = False

if PROJECT_ROOT is ...:
    raise ValueError("Replace ... in PROJECT_ROOT with your checkout path.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))


## Parse and save the datasets

The workflow parses the XML/workbook inputs and validates the supplied DPI/PPI
CSVs. It saves coverage, drug metadata, MeSH annotations, resolved DDI labels,
and interaction tables.
These files are the inputs used by the later notebooks.

When using Zheng et al.'s workbook, the code reads candidate negative pairs from
Table S6 and keeps only pairs where both drugs appear in the supplied drug–target
(DPI) table. It then removes pairs that DrugBank lists as interacting and any
pairs on the study's four-pair exclusion list. For the study inputs, this leaves
3,343 negative pairs.

If you supply a candidate-pair CSV instead, the code applies the same exclusions
to its pairs, but does not require both drugs to appear in the DPI table.

Matching completed outputs can be reused.


In [ ]:
from dtpkg.preparation import prepare_data

outputs = None
if RUN_PREPARATION:
    supplied_paths = {
        "DATA_DIR": DATA_DIR,
        "DRUGBANK_XML": DRUGBANK_XML,
        "MESH_XML": MESH_XML,
        "NEGATIVE_SOURCE": NEGATIVE_SOURCE,
        "DPI_CSV": DPI_CSV,
        "PPI_CSV": PPI_CSV,
    }
    missing = [name for name, value in supplied_paths.items() if value is ...]
    if missing:
        raise ValueError("Fill in these paths: " + ", ".join(missing))

    outputs = prepare_data(
        DATA_DIR,
        drugbank_xml=DRUGBANK_XML,
        mesh_xml=MESH_XML,
        negative_source=NEGATIVE_SOURCE,
        dpi_csv=DPI_CSV,
        ppi_csv=PPI_CSV,
    )


## Prepared datasets

The summary reports dataset sizes and relative output paths. Source and output
checksums are recorded in `preparation_manifest.json`.


In [ ]:
import json
import pandas as pd
from IPython.display import display

if outputs is not None:
    summary = json.loads(outputs["summary"].read_text())
    datasets = pd.DataFrame(summary["datasets"])
    columns = [name for name in ("dataset", "rows", "path") if name in datasets]
    display(datasets[columns])


Continue with [DrugBank exploration](../experiments/01_data_exploration/DrugBankExplore.ipynb) and
[network construction](../experiments/02_network_construction/NetworkConstruction.ipynb),
using the same `DATA_DIR`. The later tasks read the saved datasets without
requiring the original download paths. The
[MeSH-scope experiment](../experiments/03_mesh_scope/MeSHScope.ipynb) generates
the global MeSH embeddings. The inductive task creates its own holdouts and
fits new MeSH representations on development drugs only.
